# Workshop: Manual Implementation of Log-Loss (Binary Cross-Entropy)

**Course**: CSCN8010 - Statistical Machine Learning  
**Group**: 2  
**Team Members**: Ali Cihan Ozdemir, Lohith Reddy Danda

---

## Section 1: Theoretical Foundation

### 1.1 The Concept of Log-Loss (Binary Cross-Entropy)

Imagine you are a teacher grading a multiple-choice test.  
If a student answers correctly with 100% confidence, they get full marks (Loss = 0).  
If they answer incorrectly with 100% confidence, they should be penalized heavily (Loss -> $\infty$).  

**Log-Loss** acts as this penalty mechanism for classification models that output probabilities. Instead of just asking "Did the model get it right?" (Accuracy), Log-Loss asks **"How confident was the model in its prediction?"**.
- A wrong prediction made with high confidence is penalized much more severely than a wrong prediction made with uncertainty.

### 1.2 Why Log-Loss instead of Mean Squared Error (MSE)?

In Linear Regression, we use Mean Squared Error (MSE) because the cost function is convex (bowl-shaped), making it easy for Gradient Descent to find the global minimum.  

However, in **Logistic Regression**, the hypothesis function $h_\theta(x)$ is non-linear (Sigmoid). If we use MSE, the resulting cost function becomes **non-convex** (wavy with many local minima). Gradient Descent would get stuck in these local minima and fail to find the best parameters.

**Maximum Likelihood Estimation (MLE)** intuition:  
Log-Loss is derived from the principle of Maximum Likelihood. We want to find parameters $\theta$ that maximize the probability (likelihood) of observing the actual data labels. Taking the negative logarithm of this likelihood turns the product of probabilities into a sum, which is mathematically easier to minimize. This results in a **convex** cost function that guarantees finding the global minimum.

### 1.3 The Mathematical Formula

The cost function $J(\theta)$ for a single training example is defined as:

$$
Cost(h_\theta(x), y) = \begin{cases} -\log(h_\theta(x)) & \text{if } y = 1 \\ -\log(1 - h_\theta(x)) & \text{if } y = 0 \end{cases}
$$

Combining these two cases into a single formula for the entire dataset of $m$ examples:

$$
J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(h_\theta(x^{(i)})) + (1 - y^{(i)}) \log(1 - h_\theta(x^{(i)}))]
$$

Where:
- $m$ is the number of training examples.
- $y^{(i)}$ is the actual label (0 or 1).
- $h_\theta(x^{(i)})$ is the predicted probability (output of the sigmoid function).

---

## Section 2: Implementation (Python)

We will now implement the algorithm manually without relying on high-level library functions for the calculation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# 2.1 Synthetic Dataset Generation
# 'Hours Studied' vs 'Result' (0 = Fail, 1 = Pass)
hours_studied = np.array([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 8, 9, 10])
results = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1]) # 0: Fail, 1: Pass

print(f"Dataset Size: {len(hours_studied)} samples")

### 2.2 Sigmoid Function Implementation
The sigmoid function maps any real-valued number into the range [0, 1].
$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid of z.
    z: A scalar or numpy array.
    Returns: Sigmoid value between 0 and 1.
    """
    return 1 / (1 + np.exp(-z))

### 2.3 Manual Log-Loss Implementation
We implement the cost function $J(\theta)$ derived in Section 1.3.

In [ ]:
def manual_log_loss(y_true, y_pred_prob):
    """
    Compute the Log-Loss manually.
    y_true: Array of actual labels (0 or 1).
    y_pred_prob: Array of predicted probabilities (between 0 and 1).
    Returns: The average Log-Loss.
    """
    m = len(y_true)
    
    # Clipping predicted probabilities to avoid log(0) error (undefined)
    # We clip values to be between epsilon and 1-epsilon
    epsilon = 1e-15
    y_pred_prob = np.clip(y_pred_prob, epsilon, 1 - epsilon)
    
    # Calculate sum of losses
    # Formula: sum( y*log(h(x)) + (1-y)*log(1-h(x)) )
    total_cost = np.sum(y_true * np.log(y_pred_prob) + (1 - y_true) * np.log(1 - y_pred_prob))
    
    # Average cost
    average_cost = -1/m * total_cost
    
    return average_cost

### 2.4 Visualization: Log-Loss Curve
Visualizing how the penalty increases as the predicted probability deviates from the actual label.

In [ ]:
# Generate hypothetical probabilities from 0.01 to 0.99
predicted_probs = np.linspace(0.01, 0.99, 100)

# Calculate loss when actual label y = 1
loss_y1 = -np.log(predicted_probs)

# Calculate loss when actual label y = 0
loss_y0 = -np.log(1 - predicted_probs)

plt.figure(figsize=(10, 6))
plt.plot(predicted_probs, loss_y1, label='Actual Label y=1', color='blue', linewidth=2)
plt.plot(predicted_probs, loss_y0, label='Actual Label y=0', color='red', linewidth=2)
plt.title('Log-Loss (Binary Cross-Entropy) Curve')
plt.xlabel('Predicted Probability $h_\\theta(x)$')
plt.ylabel('Loss (Cost)')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

---

## Section 3: Practical Example

We will now use a simple Logistic Model to predict probabilities for our 'Hours Studied' dataset and calculate the total average Log-Loss.

In [ ]:
# Reshape data for sklearn
X = hours_studied.reshape(-1, 1)
y = results

# Train a simple Logistic Regression model
model = LogisticRegression()
model.fit(X, y)

# Predict probabilities for the training set
y_prob = model.predict_proba(X)[:, 1]

print("Hours Studied | Actual Result | Predicted Prob")
print("-"*45)
for i in range(len(X)):
    print(f"{X[i][0]:^13} | {y[i]:^13} | {y_prob[i]:.4f}")

In [ ]:
# Calculate Log-Loss using our Manual Implementation
manual_loss_value = manual_log_loss(y, y_prob)

# Calculate Log-Loss using Sklearn's built-in function for verification
sklearn_loss_value = log_loss(y, y_prob)

print(f"\nManual Log-Loss Calculation:   {manual_loss_value:.10f}")
print(f"Sklearn Log-Loss Calculation:  {sklearn_loss_value:.10f}")

difference = abs(manual_loss_value - sklearn_loss_value)
print(f"Difference: {difference:.10e}")

if difference < 1e-10:
    print("\n✅ SUCCESS: Manual implementation matches Sklearn's result!")
else:
    print("\n❌ FAILURE: Significant discrepancy found.")

---

## Section 4: Talking Points for Presentation

### 1. Penalty for "Confident but Wrong" Predictions
Log-Loss is extremely sensitive to confident errors. If the model predicts a probability of 0.99 for a positive class, but the actual label is negative, the loss is enormous. This property forces the model to be honest about its uncertainty and calibrates the probabilities effectively.

### 2. Convexity ensures Global Minimum
Unlike Mean Squared Error (MSE), which creates a non-convex "wavy" error surface when applied to the sigmoid function, Log-Loss creates a convex "bowl-shaped" error surface. This ensures that gradient descent will always converge to the global minimum (best possible parameters) rather than getting stuck in a suboptimal local minimum.

### 3. Role in Gradient Descent & Maximum Likelihood
The gradient of the Log-Loss function with respect to the weights turns out to be mathematically elegant: $(h_\theta(x) - y)x$. This simple derivative looks surprisingly similar to linear regression's gradient, making the update steps in gradient descent computationally efficient while statistically sound (based on Maximum Likelihood Estimation).